In [ ]:
import anndata as ad
import gseapy as gs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pertpy as pt
import scanpy as sc
import seaborn as sns
from scxmatch import *
from statsmodels.stats.multitest import multipletests

In [ ]:
np.random.seed(43)

In [ ]:
group_by = "perturbation"
reference = "control"

In [ ]:
hallmark = gs.get_library('MSigDB_Hallmark_2020')

In [ ]:
adata = ad.read_h5ad("../data/LSI_MimitouSmibert2021.hdf5") # this is not log1p transformed and contains all genes
adata = adata[adata.obs["perturbation"].dropna().index].copy()
#adata.obs[group_by] = adata.obs[group_by].astype(str)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata)
adata.var.rename({"Unnamed: 0": "gene"}, axis=1, inplace=True)
adata.var.set_index("gene", inplace=True)

In [ ]:
test_groups = adata.obs["perturbation"].unique()

In [ ]:
group_counts = adata.obs[group_by].value_counts()
min_count = group_counts.min()

print(f"Minimum group size: {min_count}", flush=True)

sampled_indices = []
relative_support_dict = dict()

for g in group_counts.index:
    idx = np.where(adata.obs[group_by] == g)[0]
    sampled = np.random.choice(idx, min_count, replace=False)
    sampled_indices.append(sampled)
    relative_support_dict[g] = len(sampled) / len(idx)

sampled_indices = np.concatenate(sampled_indices)
adata_balanced = adata[sampled_indices].copy()

In [ ]:
n_comps = min(50, adata_balanced.shape[1] - 1, adata_balanced.shape[0] - 1)
sc.pp.pca(adata_balanced, n_comps=n_comps)
distance_test = pt.tl.DistanceTest("edistance", n_perms=10000, obsm_key="X_pca")
tab = distance_test(adata_balanced, groupby="perturbation", contrast="control")
tab

In [ ]:
n_comps = min(50, adata_balanced.shape[1] - 1, adata_balanced.shape[0] - 1)
distance_test = pt.tl.DistanceTest("edistance", n_perms=20000, obsm_key="X_pca")
tab = distance_test(adata_balanced, groupby="perturbation", contrast="control")
tab

In [ ]:
n_comps = min(50, adata.shape[1] - 1, adata.shape[0] - 1)
sc.pp.pca(adata, n_comps=n_comps)
distance_test = pt.tl.DistanceTest("edistance", n_perms=10000, obsm_key="X_pca")
tab = distance_test(adata, groupby="perturbation", contrast="control")
tab

In [ ]:
n_comps = min(50, adata.shape[1] - 1, adata.shape[0] - 1)
distance_test = pt.tl.DistanceTest("edistance", n_perms=20000, obsm_key="X_pca")
tab = distance_test(adata, groupby="perturbation", contrast="control")
tab

In [ ]:
adata = ad.read_h5ad("../data/LSI_MimitouSmibert2021.hdf5") # this is not log1p transformed and contains all genes
key = 'TNF-alpha Signaling via NF-kB'
name = 'MSigDB_Hallmark_2020'

adata = adata[adata.obs["perturbation"].dropna().index].copy()
sc.pp.log1p(adata)
#sc.pp.highly_variable_genes(adata)
adata.var.rename({"Unnamed: 0": "gene"}, axis=1, inplace=True)
adata.var.set_index("gene", inplace=True)
subset = adata[:, np.isin(adata.var_names, hallmark[key])].copy()

In [ ]:
rows = list()

In [ ]:
for test_group in test_groups:
    if test_group != "control":
        results = test(adata=subset, reference=reference, test_group=test_group, k="full", metric="sqeuclidean", rank=False, group_by=group_by)
        p = results["p_value"]
        z = results["z_score"]
        s = results["relative_support"]
        rows.append([f"{name} {key}", subset.shape[1], len(hallmark[key]), test_group, p, z, s])

In [ ]:
results = pd.DataFrame(rows, columns=["Gene set", "#Genes used", "#Genes total", "Test group (knockout)", "P", "Z", "S"])
results["P_adj"] = multipletests(results["P"], method="fdr_bh")[1]

In [ ]:
group_counts = subset.obs[group_by].value_counts()
min_count = group_counts.min()

print(f"Minimum group size: {min_count}", flush=True, file=sys.stderr)

sampled_indices = []
relative_support_dict = dict()

for g in group_counts.index:
    idx = np.where(subset.obs[group_by] == g)[0]
    sampled = np.random.choice(idx, min_count, replace=False)
    sampled_indices.append(sampled)
    relative_support_dict[g] = len(sampled) / len(idx)

sampled_indices = np.concatenate(sampled_indices)

In [ ]:
subset_balanced = subset[sampled_indices].copy()

In [ ]:
n_comps = min(50, subset.shape[1] - 1, subset.shape[0] - 1)
sc.pp.pca(subset_balanced, n_comps=n_comps)
distance_test = pt.tl.DistanceTest("edistance", n_perms=10000, obsm_key="X_pca")
tab = distance_test(subset_balanced, groupby="perturbation", contrast="control")

In [ ]:
pd.options.display.float_format = "{:,.1e}".format
results.sort_values("Test group (knockout)")

In [ ]:
tab